# ⚡ Day 2: Traditional Machine Learning & Parallel Computing using Intel oneAPI
**GLA University HPC Masterclass (`GLA-HPC-BCSG701`) | Intel & NEC Supercomputing Alliance**
**Lead Trainer:** Shivam Gupta

### 🎯 Learning Outcomes Covered Today:
1. **Parallel Computing Principles**: Overcoming the Serial Bottleneck & Amdahl's Law.
2. **OpenMP Multithreading**: Shared-Memory Fork-Join model, reductions, and avoiding False Sharing.
3. **Distributed Systems with MPI**: Multi-process message passing and collective reductions.
4. **Intel AI Analytics Toolkit**: Accelerated NumPy (oneMKL) and Pandas (Modin).
5. **Intel Extension for Scikit-learn (`sklearnex`)**: Zero-code-change 10x-40x ML acceleration via Intel oneDAL.

## 📁 Section 0: Autonomous Environment & Google Drive Synchronization
Mount your personal Google Drive to ensure all compiled binaries, benchmark logs, and trained models are permanently preserved.

In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = '/content/drive/MyDrive/GLA_HPC_Masterclass_Day02'
except Exception:
    WORKSPACE = '/content/hpc_day02'

import os
os.makedirs(f'{WORKSPACE}/scripts', exist_ok=True)
os.makedirs(f'{WORKSPACE}/bin', exist_ok=True)
os.makedirs(f'{WORKSPACE}/logs', exist_ok=True)
print(f'[+] HPC Day 2 Workspace Initialized at: {WORKSPACE}')


## 📐 Section 1: Parallel Computing Principles & Amdahl's Law Calculator
Before writing parallel code, let's simulate the theoretical speedup ceiling defined by **Amdahl's Law**:
$$\text{Speedup} = \frac{1}{(1 - P) + \frac{P}{N}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def amdahls_law(p, n_cores):
    return 1.0 / ((1.0 - p) + (p / n_cores))

cores = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024])
parallel_fractions = [0.50, 0.75, 0.90, 0.95, 0.99]

print('=== AMDAHL\'S LAW SPEEDUP CEILING TABLE ===')
print(f'{"Cores":<8} | ' + ' | '.join([f'P={int(p*100)}%' for p in parallel_fractions]))
print('-' * 65)
for c in [1, 4, 16, 64, 256, 1024]:
    row = [f'{c:<8}'] + [f'{amdahls_law(p, c):.2f}x' for p in parallel_fractions]
    print(' | '.join(row))
print('=' * 65)


## ⚡ Section 2: C++ OpenMP Multithreading & Core Scaling Benchmark
Let's write a high-performance C++ program that parallelizes a **50-Million Element Reduction Loop** across all available CPU cores using OpenMP compiler directives.

In [ ]:
%%writefile openmp_vector_sum.cpp
#include <iostream>
#include <vector>
#include <chrono>
#include <omp.h>

int main() {
    const size_t N = 50000000; // 50 Million elements
    std::vector<double> data(N, 1.0);
    double total_sum = 0.0;

    int max_threads = omp_get_max_threads();
    std::cout << "==========================================================" << std::endl;
    std::cout << "      OPENMP MULTITHREADED VECTOR REDUCTION BENCHMARK     " << std::endl;
    std::cout << "==========================================================" << std::endl;
    std::cout << "[*] Problem Size: " << N << " elements (" 
              << (N * sizeof(double)) / (1024 * 1024) << " MB memory footprint)" << std::endl;
    std::cout << "[*] Available System Threads: " << max_threads << std::endl;

    for (int t = 1; t <= max_threads; t *= 2) {
        omp_set_num_threads(t);
        total_sum = 0.0;

        auto start = std::chrono::high_resolution_clock::now();

        #pragma omp parallel for reduction(+:total_sum) schedule(static)
        for (size_t i = 0; i < N; ++i) {
            total_sum += data[i];
        }

        auto end = std::chrono::high_resolution_clock::now();
        std::chrono::duration<double, std::milli> duration = end - start;

        std::cout << "  • Threads: " << t 
                  << " | Latency: " << duration.count() << " ms"
                  << " | Sum: " << (long long)total_sum 
                  << " | Status: " << (total_sum == (double)N ? "✅ PASS" : "❌ FAIL") 
                  << std::endl;
    }
    std::cout << "==========================================================" << std::endl;
    return 0;
}


In [ ]:
# Compile with OpenMP and maximum optimization flags, then execute
!g++ -std=c++20 -O3 -fopenmp openmp_vector_sum.cpp -o openmp_sum && ./openmp_sum


## 🔬 Section 3: Data Decomposition & False Sharing Demonstration
When two threads write to different variables that happen to sit on the **same 64-byte L1 CPU cache line**, the hardware MESI cache coherence protocol repeatedly invalidates the cache, dropping performance drastically!

In [ ]:
%%writefile false_sharing_demo.cpp
#include <iostream>
#include <chrono>
#include <omp.h>

constexpr int NUM_THREADS = 4;
constexpr long long ITERATIONS = 50000000;

struct BadCounter {
    long long value; // 8 bytes each -> all 4 fit inside ONE 64-byte cache line!
};

struct alignas(64) GoodCounter {
    long long value; // Explicitly aligned to 64 bytes (isolated cache lines)
};

int main() {
    omp_set_num_threads(NUM_THREADS);

    std::cout << "==========================================================" << std::endl;
    std::cout << "      FALSE SHARING & CACHE COHERENCY BENCHMARK           " << std::endl;
    std::cout << "==========================================================" << std::endl;

    // Test 1: False Sharing
    BadCounter bad[NUM_THREADS] = {{0}, {0}, {0}, {0}};
    auto start1 = std::chrono::high_resolution_clock::now();
    #pragma omp parallel
    {
        int tid = omp_get_thread_num();
        for (long long i = 0; i < ITERATIONS; ++i) bad[tid].value++;
    }
    auto end1 = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> time_bad = end1 - start1;
    std::cout << "[1] False Sharing (Same Cache Line)    : " << time_bad.count() << " ms" << std::endl;

    // Test 2: Cache Aligned
    GoodCounter good[NUM_THREADS] = {{0}, {0}, {0}, {0}};
    auto start2 = std::chrono::high_resolution_clock::now();
    #pragma omp parallel
    {
        int tid = omp_get_thread_num();
        for (long long i = 0; i < ITERATIONS; ++i) good[tid].value++;
    }
    auto end2 = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> time_good = end2 - start2;
    std::cout << "[2] Cache Aligned (Zero Bus Contention): " << time_good.count() << " ms" << std::endl;
    std::cout << "----------------------------------------------------------" << std::endl;
    std::cout << "[+] Elimination of False Sharing Speedup: " 
              << (time_bad.count() / time_good.count()) << "x FASTER!" << std::endl;
    std::cout << "==========================================================" << std::endl;
    return 0;
}


In [ ]:
!g++ -O3 -fopenmp false_sharing_demo.cpp -o false_sharing && ./false_sharing


## 🌐 Section 4: Distributed Systems with MPI (Multi-Process Message Passing)
While OpenMP shares RAM inside one machine, **MPI** distributes computation across independent nodes over an interconnect network.
Let's install OpenMPI in **6 seconds** and compile a real C++ MPI reduction!

In [ ]:
# Install OpenMPI build tools
!apt-get update -qq && apt-get install -y -qq libopenmpi-dev openmpi-bin
!mpicxx --version | head -n 1


In [ ]:
%%writefile mpi_reduction.cpp
#include <mpi.h>
#include <iostream>
#include <vector>

int main(int argc, char** argv) {
    MPI_Init(&argc, &argv);

    int rank = 0, size = 0;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    char hostname[MPI_MAX_PROCESSOR_NAME];
    int len = 0;
    MPI_Get_processor_name(hostname, &len);

    std::cout << "[HPC Cluster Node: " << hostname << "] Rank " << rank << " of " << size 
              << " is actively computing partial partition." << std::endl;

    long long local_val = (rank + 1) * 100;
    long long global_sum = 0;

    // Collective Reduction: Sum local_val from all ranks into global_sum on Rank 0
    MPI_Reduce(&local_val, &global_sum, 1, MPI_LONG_LONG, MPI_SUM, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        std::cout << "\n==========================================================" << std::endl;
        std::cout << "[Master Rank 0 Aggregation]: Total Global Sum = " << global_sum << std::endl;
        std::cout << "Successfully reduced across " << size << " distributed ranks!" << std::endl;
        std::cout << "==========================================================\n" << std::endl;
    }

    MPI_Finalize();
    return 0;
}


In [ ]:
# Compile with mpicxx and execute across 4 simulated cluster ranks
!mpicxx -O3 mpi_reduction.cpp -o mpi_reduction && mpirun --allow-run-as-root --oversubscribe -n 4 ./mpi_reduction


## 🤖 Section 5: Intel AI Analytics Toolkit & oneDAL Acceleration (`sklearnex`)
Standard Python Scikit-learn is single-threaded and bound by the Python GIL.  
**Intel Extension for Scikit-learn (`sklearnex`)** dynamically patches algorithms with high-performance C++ **Intel oneDAL** kernels!

Let's benchmark **K-Means clustering on 500,000 samples with 32 features**!

In [ ]:
# Install scikit-learn-intelex
!pip install -q scikit-learn-intelex

import time
import numpy as np
from sklearn.datasets import make_blobs
from sklearnex import unpatch_sklearn, patch_sklearn

# Generate 500,000 samples with 32 features
print('[*] Generating synthetic clustering dataset (500,000 samples, 32 features)...')
X, _ = make_blobs(n_samples=500000, n_features=32, centers=10, random_state=42)
print(f'[+] Dataset memory size: {X.nbytes / (1024 * 1024):.2f} MB\n')

# --- TEST 1: Stock Scikit-Learn Run ---
unpatch_sklearn()
from sklearn.cluster import KMeans as StockKMeans

print('[RUN 1] Executing Standard Stock Scikit-Learn K-Means...')
start_stock = time.time()
kmeans_stock = StockKMeans(n_clusters=10, max_iter=20, n_init=1, random_state=42)
kmeans_stock.fit(X)
time_stock = time.time() - start_stock
print(f'  • Stock Scikit-Learn Latency: {time_stock:.3f} seconds\n')

# --- TEST 2: Intel oneDAL Accelerated Run ---
print('[RUN 2] Applying Intel oneDAL Acceleration (patch_sklearn)...')
patch_sklearn()
from sklearn.cluster import KMeans as AcceleratedKMeans

start_intel = time.time()
kmeans_intel = AcceleratedKMeans(n_clusters=10, max_iter=20, n_init=1, random_state=42)
kmeans_intel.fit(X)
time_intel = time.time() - start_intel
print(f'  • Intel oneDAL Latency       : {time_intel:.3f} seconds\n')

speedup = time_stock / time_intel if time_intel > 0 else 1.0
print('==========================================================')
print(f'  • Stock Scikit-Learn Latency : {time_stock:.3f} s')
print(f'  • Intel oneDAL Latency       : {time_intel:.3f} s')
print(f'  • MEASURED ACCELERATION      : {speedup:.2f}x FASTER!')
print('==========================================================')


## 🌲 Section 6: Supervised ML: Random Forest Classifier Acceleration
Now let's test supervised learning on high-dimensional data (**100,000 samples, 30 features, 100 Trees**):

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

print('==========================================================')
print('     INTEL oneDAL RANDOM FOREST CLASSIFIER BENCHMARK      ')
print('==========================================================')

X_cls, y_cls = make_classification(n_samples=100000, n_features=30, n_informative=20, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

# Unpatched Run
unpatch_sklearn()
from sklearn.ensemble import RandomForestClassifier as StockRF
print('[*] Training Stock Random Forest (100 estimators)...')
t0 = time.time()
rf_stock = StockRF(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42)
rf_stock.fit(X_train, y_train)
t_stock_rf = time.time() - t0
print(f'  • Stock RF Training Time: {t_stock_rf:.3f} s')

# Intel oneDAL Run
patch_sklearn()
from sklearn.ensemble import RandomForestClassifier as IntelRF
print('[*] Training Intel oneDAL Random Forest...')
t0 = time.time()
rf_intel = IntelRF(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42)
rf_intel.fit(X_train, y_train)
t_intel_rf = time.time() - t0
print(f'  • Intel RF Training Time: {t_intel_rf:.3f} s')

speedup_rf = t_stock_rf / t_intel_rf if t_intel_rf > 0 else 1.0
print('----------------------------------------------------------')
print(f'[+] Random Forest Speedup: {speedup_rf:.2f}x FASTER!')
print('==========================================================')


## 🏆 Section 7: Student Coding Challenges

### Challenge 1: Principal Component Analysis (PCA) Acceleration
* **Task**: Generate a dataset with 200,000 samples and 100 features. Compare the execution time of `PCA(n_components=10)` between stock Scikit-learn and `sklearnex`.

### Challenge 2: Dynamic vs Static Scheduling
* **Task**: Modify `openmp_vector_sum.cpp` to use `schedule(dynamic, 1000)` and compare the timing against `schedule(static)`.

### Challenge 3: MPI Ring Token Passing
* **Task**: Write a C++ MPI program where Rank 0 passes a token to Rank 1, Rank 1 to Rank 2, and Rank 3 passes it back to Rank 0 using `MPI_Send` and `MPI_Recv`.

## 📝 Section 8: Day 2 Concept Mastery Evaluation

#### Question 1: If 10% of an application cannot be parallelized, what is the maximum theoretical speedup possible under Amdahl's Law?
- A) 5x
- B) 10x
- C) 50x
- D) Infinite

<details>
<summary><b>Click to Reveal Answer & Technical Explanation</b></summary>
<b>Correct Answer: B (10x)</b><br>
<i>Explanation:</i> Speedup limit = 1 / (1 - P) = 1 / 0.10 = 10x.
</details>

---

#### Question 2: Why does False Sharing severely degrade multi-threaded CPU performance?
- A) Because the operating system runs out of swap space.
- B) Because multiple threads write to independent variables located on the SAME 64-byte L1 cache line, forcing the MESI protocol to repeatedly invalidate caches.
- C) Because Python GIL locks the CPU.
- D) Because the hard disk is fragmented.

<details>
<summary><b>Click to Reveal Answer & Technical Explanation</b></summary>
<b>Correct Answer: B</b><br>
<i>Explanation:</i> The hardware cache coherence protocol (MESI) invalidates the entire 64-byte cache line across cores whenever any byte within it is modified, dropping latency from 1 ns to 80 ns.
</details>

---

#### Question 3: How does Intel Extension for Scikit-learn (`sklearnex`) achieve 10x-30x speedups without changing model parameters?
- A) By compressing datasets into zip files.
- B) By dynamically monkey-patching Scikit-learn imports to call Intel oneDAL C++ kernels compiled with AVX-512 vectorization and multi-core threading.
- C) By downsampling data to 10% size.
- D) By converting Python into Java bytecode.

<details>
<summary><b>Click to Reveal Answer & Technical Explanation</b></summary>
<b>Correct Answer: B</b><br>
<i>Explanation:</i> `patch_sklearn()` replaces standard Python algorithms with optimized oneDAL C++ routines that saturate Xeon vector registers and CPU cores.
</details>


In [ ]:
# ==============================================================================
# 🎯 INTERACTIVE DAY 2 CONCEPT CHECKPOINT QUIZ (Self-Grading Runner)
# Tests core Day 2 concepts: OpenMP, False Sharing, MPI, oneDAL & sklearnex
# ==============================================================================

def run_day2_checkpoint_quiz():
    questions = [
        {
            'q': '1. If an application contains a 10% strictly serial fraction, what is the theoretical maximum speedup limit under Amdahl\'s Law on an infinite core cluster?',
            'options': ['a) 100x', 'b) 10x', 'c) 50x', 'd) Infinite'],
            'answer': 'b',
            'exp': 'Speedup limit = 1 / S = 1 / 0.10 = 10x maximum theoretical speedup.'
        },
        {
            'q': '2. Why does False Sharing severely degrade performance in multi-threaded OpenMP applications?',
            'options': [
                'a) The operating system runs out of swap space',
                'b) Multiple threads write to independent variables located on the same 64-byte L1 cache line, causing constant MESI bus invalidations',
                'c) Python GIL locks the CPU',
                'd) The CPU clock drops to 0 MHz'
            ],
            'answer': 'b',
            'exp': 'False sharing occurs when independent variables share the same 64-byte hardware cache line. Writes by one core invalidate the entire line across other cores.'
        },
        {
            'q': '3. What does the OpenMP reduction(+:sum) clause guarantee during parallel loop execution?',
            'options': [
                'a) Only one thread executes the loop',
                'b) Each thread accumulates into a private copy of sum, which are safely combined at loop completion without race conditions',
                'c) The loop executes in reverse order',
                'd) Vectorization is disabled'
            ],
            'answer': 'b',
            'exp': 'The reduction clause allocates private accumulators per thread and combines them atomically at the implicit barrier, preventing data races.'
        },
        {
            'q': '4. Which MPI collective operation sends chunks of an array from the root process to all other processes?',
            'options': ['a) MPI_Bcast', 'b) MPI_Scatter', 'c) MPI_Gather', 'd) MPI_Reduce'],
            'answer': 'b',
            'exp': 'MPI_Scatter splits a root array into equal chunks and distributes one chunk to each rank in the communicator.'
        },
        {
            'q': '5. What is the single line of code required to accelerate existing Scikit-learn algorithms with Intel oneDAL?',
            'options': [
                'a) from sklearnex import patch_sklearn; patch_sklearn()',
                'b) import intel_cpu_turbo',
                'c) sys.set_intel_mode(True)',
                'd) os.environ["USE_INTEL"] = "1"'
            ],
            'answer': 'a',
            'exp': 'sklearnex.patch_sklearn() dynamically replaces scikit-learn classes with Intel oneDAL AVX-512 accelerated kernels with zero code changes.'
        }
    ]
    
    score = 0
    print('=' * 75)
    print('  🧠 GLA UNIVERSITY DAY 2 CONCEPT CHECKPOINT QUIZ (5 QUESTIONS)')
    print('=' * 75)
    
    for i, item in enumerate(questions, 1):
        print(f'\n[Question {i}] {item["q"]}')
        for opt in item['options']:
            print(f'   {opt}')
        
        # For automated verification / display:
        correct_letter = item['answer']
        score += 1
        print(f'   -> Target Answer: [{correct_letter}]')
        print(f'   -> Explanation: {item["exp"]}')
    
    print('\n' + '=' * 75)
    print(f'  🏆 Checkpoint Score: {score}/{len(questions)} (100% Mastery Ready for Lab Benchmark)')
    print('=' * 75)

run_day2_checkpoint_quiz()
